In [ ]:
import sys
import langchain_community.vectorstores

# Creamos un objeto falso para que LangChain crea que sí existe
class FakeDatabricks:
    pass

# Lo inyectamos directamente en la librería en tiempo de ejecución
sys.modules['langchain_community.vectorstores'].DatabricksVectorSearch = FakeDatabricks
langchain_community.vectorstores.DatabricksVectorSearch = FakeDatabricks

# ADVERTENCIA: Pon tus imports normales DEBAJO de estas líneas
# /Users/gblasd/Documents/SmartBnB/.venv/lib/python3.13/site-packages/langchain_classic/retrievers/self_query/base.py
# line 72 updated

In [2]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain
from langchain_chroma import Chroma

from uuid import uuid4
import pandas as pd
import chromadb


import psycopg2
import os
import datetime, uuid
from datetime import datetime
from decimal import Decimal

/Users/gblasd/Documents/SmartBnB/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize the OpenAI embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a ChromDB vector database
vector_store = Chroma(
    collection_name='smartbnb_vector_store',
    embedding_function=embeddings,
    persist_directory='/Users/gblasd/Documents/SmartBnB/db/chroma_db'
)

In [6]:
client = chromadb.PersistentClient(path='/Users/gblasd/Documents/SmartBnB/db/chroma_db')

In [7]:
client.list_collections()

[Collection(name=smartbnb_vector_store), Collection(name=example_collection)]

In [6]:
# client.get_collection(name="example_collection")
# client.delete_collection('example_collection')

In [7]:
# Create connection to the database and initialize it
def create_db_connection() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5433"),
        dbname=os.getenv("DB_NAME", "smartbnb"),
        user=os.getenv("DB_USER", "admin"),
        password=os.getenv("DB_PASSWORD", "admin")
    )
    return conn

def drop_connection(conn):
    conn.close()

conn = create_db_connection()

def _sanitize_metadata_value(v):

    # to contain only str, int, float, or bool
    if isinstance(v, (str, int, float, bool)):
        return v
    elif v is None:
        return None
    elif isinstance(v, (Decimal,)):
        return float(v)
    elif isinstance(v, (list, tuple)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, dict):
        return {k: _sanitize_metadata_value(v) for k, v in v.items()}
    elif isinstance(v, (set, frozenset)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, (np.ndarray,)):
        return v.tolist()
    elif isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()
    elif isinstance(v, uuid.UUID):
        return str(v)
    return v


def extract_data_from_pg():

    documents = []
    id_documents = []

    # Query the database to get all records from the listings table
    with conn.cursor() as cur:
        cur.execute("""select l.id, l.listing_url, l.name, l.description, l.neighborhood_overview, l.neighbourhood_cleansed,
        l.property_type, l.room_type, l.accommodates, l.bathrooms, l.bathrooms_text, l.bedrooms, 
        l.beds, l.amenities, l.price, l.latitude, l.longitude, l.minimum_nights, l.maximum_nights, 
        l.has_availability, l.review_scores_accuracy, l.review_scores_communication,
        l.review_scores_cleanliness, l.review_scores_location, l.review_scores_value, 
        l.review_scores_rating, l.reviews_per_month, l.instant_bookable,
        l.calculated_host_listings_count, l.calculated_host_listings_count_entire_homes,
        l.calculated_host_listings_count_private_rooms, l.calculated_host_listings_count_shared_rooms
    from public.listings l
    where l.has_availability is true""")
        records = cur.fetchall()
        for record in records:
            # Add the record to the vector database collection
            row = {}
            row["metadata"] = [
                {
                    col.name: _sanitize_metadata_value(record[i]) 
                    for i, col in enumerate(cur.description) 
                    if col.name  in ["neighbourhood_cleansed","property_type","room_type", "bathrooms", "bathrooms_text", "bedrooms", "beds",
                                    "price", "latitude", "longitude", "minimum_nights", "maximum_nights", "has_availability", 
                                    "review_scores_accuracy", "amenities"]
                }
            ]

            # Convert amenities from string to list
            if "amenities" in row["metadata"][0]:
                amenities_str = row["metadata"][0]["amenities"]
                amenities_list = [amenity.strip() for amenity in amenities_str.split(",")]
                row["metadata"][0]["amenities"] = amenities_list

            # create Document object            
            doc = Document(
                page_content=str(record[3]),
                metadata=row["metadata"][0],
                id=record[0]
            )

            # Add Document object to python list
            documents.append(doc)
            id_documents.append(str(record[0]))
            
            # vector_store.add(
            #     ids=str(record[0]),
            #     documents=str(record[3]),
            #     metadatas=row["metadata"]
            # )

    drop_connection(conn)

    return documents, id_documents

# q_docs, q_id = extract_data_from_pg()

# len(q_docs), len(q_id)

# Indexing

In [8]:
# # Define a safe batch size
# BATCH_SIZE = 5000

# # Loop and add documents in smaller chunks
# for i in range(0, len(q_docs), BATCH_SIZE):
#     batch_docs = q_docs[i : i + BATCH_SIZE]
#     batch_ids = q_id[i : i + BATCH_SIZE]

#     vector_store.add_documents(
#         documents=batch_docs, 
#         ids=batch_ids
#     )


## Query data

In [8]:
query_results = vector_store.similarity_search(
    query="near from the university with roof garden",
    k=10,
    filter={'neighbourhood_cleansed':'Tlalpan'}
)

In [9]:
query_results

[]

In [10]:
json_query = {
                "$and": [
                    # {
                    #     "price": {
                    #         "$gte": 1000 
                    #     }
                    # }, 
                    {
                        "price": {
                            "$lte": 3000 
                        }
                    },
                    {
                        "property_type": "Entire serviced apartment"
                    },
                    {
                        "neighbourhood_cleansed": "Tlalpan"
                    },
                    {
                        "$or": [
                            {
                                "amenities" : {
                                    "$contains": "Wine glasses"
                                }
                            },
                             {
                                "amenities" : {
                                    "$contains": 'Pets allowed'
                                }
                            },
                        ]
                    }
                ]
            }

In [11]:
json_query

{'$and': [{'price': {'$lte': 3000}},
  {'property_type': 'Entire serviced apartment'},
  {'neighbourhood_cleansed': 'Tlalpan'},
  {'$or': [{'amenities': {'$contains': 'Wine glasses'}},
    {'amenities': {'$contains': 'Pets allowed'}}]}]}

In [12]:
query_results = vector_store.similarity_search(
    query="near from the university with roof garden",
    k=10,
    filter=json_query
)

In [13]:
query_results

[]

In [15]:
data = [
    {
        "page_content": doc.page_content,
        "id": getattr(doc, "id", None),
        **(doc.metadata or {})
    }
    for doc in query_results
]

df = pd.DataFrame(data)

In [16]:
df

,page_content,id,amenities,beds,has_availability,bathrooms_text,latitude,maximum_nights,review_scores_accuracy,minimum_nights,neighbourhood_cleansed,longitude,property_type,price,bathrooms,room_type,bedrooms
0,Enjoy the simplicity of this quiet and central...,632058760451122976,"[Shampoo, Carbon monoxide alarm, Pocket wifi, ...",2.0,True,1 bath,19.29,1125.0,4.96,2.0,Tlalpan,-99.17,Entire serviced apartment,1478.0,1.0,Entire home/apt,1.0
1,Pavo Real Apartment WiFi + Amenities + Exclusi...,1148915308188077748,"[Shampoo, Carbon monoxide alarm, Dedicated wor...",5.0,True,2 baths,19.31,365.0,5.00,2.0,Tlalpan,-99.13,Entire serviced apartment,2511.0,2.0,Entire home/apt,3.0
2,The apartment is located within the Torres Tla...,43987492,"[Free parking on premises \u2013 1 space, Carb...",4.0,True,1 bath,19.28,1125.0,4.87,1.0,Tlalpan,-99.16,Entire serviced apartment,1369.0,1.0,Entire home/apt,3.0
3,"Escape the noise, pollution and crowds to an e...",979993910630455469,"[Shampoo, Electric stove, Conditioner, Dedicat...",1.0,True,1 bath,19.23,365.0,4.00,3.0,Tlalpan,-99.17,Entire serviced apartment,708.0,1.0,Entire home/apt,1.0


# Retriever

**Advanced Retriever Configuration**

In [14]:
retriever = vector_store.as_retriever(
    # search_type="similarity_score_threshold", 
    # search_kwargs={
    #     "k": 5,  # Return the top 5 most relevant documents
    #     "score_threshold": 0.7  # Only return documents with a similarity score of 0.7 or higher
    # }
)

In [15]:
# vector store retrives with documents
retriever.vectorstore.similarity_search(
    query="near from the university with roof garden",
    k=2,
    filter=json_query
)

[]

In [16]:
data = [
    {
        "page_content": doc.page_content,
        "id": getattr(doc, "id", None),
        **(doc.metadata or {})
    }
    for doc in query_results
]

df = pd.DataFrame(data)

In [17]:
df

""


# Gnerating LLM Predictions Using Relevant Documentss

In [21]:
# inittialize prompt
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the follow context:
    {context}
    
    Question: {question}"""
)

# Initialize interface to act as our LLM
llm = ChatOpenAI(
    name='gpt-3.5-turbo', 
    temperature=0 # to eliminate the creativity in outputs from the model
    )

# combie them with the operator |
chain = prompt | llm

    # fetch relevant documents
docs = retriever.vectorstore.similarity_search(
    query="I am founding listings near from the university with roof garden, give me the best listings",
    k = 3,
    filter = json_query
)

# run, we invoke the chain passing in the context variable (our retrieved relevant docs)
chain.invoke({"context": docs, "question":"I am founding listings near from the university with roof garden, give me the best listings"})


AIMessage(content="Based on the provided context, the best listing near the university with a roof garden is likely the second listing with the ID '632058760451122976'. This listing is an entire serviced apartment located in Tlalpan, with a private terrace and access to common areas including a large garden. It has 1 bedroom, 1 bathroom, and 2 beds. The amenities include a private patio or balcony, outdoor dining area, and outdoor furniture, which could potentially include a roof garden.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 1499, 'total_tokens': 1599, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Do0uXRPMZyA9tdErcQ9IdxXZ8YSHD', 'service_tier': 'default', 'finish_reason': 

## Query transformation

_Query transformation_ is a subset of strategies designed tto modify the user´s input to answer the first RAG problem question: How we handle the variablity in the queality of a user's input?

In [121]:
rewrite_prompt = ChatPromptTemplate.from_template("""Provide a better search
query for web search engine to answer the given question, end the queries
with ’**’. Question: {x} Answer:""")

def parse_rewriter_output(message):
    return message.content.strip('"').strip("**")

# inittialize prompt
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the follow context:
    {context}
    
    Question: {question}"""
)

# Initialize interface to act as our LLM
llm = ChatOpenAI(
    name='gpt-3.5-turbo', 
    temperature=0 # to eliminate the creativity in outputs from the model
    )

# combie them with the operator |
chain = prompt | llm
rewriter = rewrite_prompt | llm | parse_rewriter_output
user_prompt = "I am founding listings near from the university with roof garden, give me the best listings"

# fetch relevant documents
docs = retriever.vectorstore.similarity_search(
    query=rewriter.invoke(user_prompt),
    k = 3,
    filter = json_query
)

# run, we invoke the chain passing in the context variable (our retrieved relevant docs)
chain.invoke({"context": docs, "question":user_prompt})

AIMessage(content="Based on the provided context, the best listing near the university with a roof garden is the second listing with ID '632058760451122976'. This listing is located in Tlalpan, has a private terrace, and access to common areas including a large garden.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 1500, 'total_tokens': 1555, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DnyYQDtTKr8H5S4ks80L7WsJOe5eS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ea024-9991-7bb1-96c2-1f3d4987eb52-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1500, 'output_tokens': 55, 'total_tokens': 155

# Query construction

_Query construction_ is the process of transforming natural language query into the query language of the database or datasource you are interaction with.

__Text-to-Metadata Filter__

In [3]:
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import ChatOpenAI

from langchain_classic.vectorstores.base import VectorStore

In [5]:
fields = [
    AttributeInfo(
        name="price",
        description="price per night of the listing",
        type="integer"
    ),
    AttributeInfo(
        name="property_type",
        description="Type of property",
        type="string"
    ),
    AttributeInfo(
        name="room_type",
        description="Type of room",
        type="string"
    ), 
    AttributeInfo(
        name="neighbourhood_cleansed",
        description="Tneighbourhood where the listing is",
        type="string"
    ), 
    AttributeInfo(
        name="review_scores_accuracy",
        description="A 1-5 rating for the listing",
        type="float"
    ),
]

description = "Brief summary of a listing"

llm_model = ChatOpenAI(temperature=0)

retriever = SelfQueryRetriever.from_llm(
    llm_model,
    vector_store,
    description,
    fields
)

print(retriever.invoke("Can you show me 5 listings near from the university with roof garden, price between 1500 and 3000, with location neighbourhood in Tlalpan", ))

[]


In [ ]:
"""
[Document(id='13084124', metadata={'review_scores_accuracy': 4.88, 'maximum_nights': 1125.0, 'longitude': -99.24, 'has_availability': True, 'price': 2625.0, 'property_type': 'Tiny home', 'amenities': ['Shampoo', 'Carbon monoxide alarm', 'Dedicated workspace', 'First aid kit', 'Indoor fireplace: wood-burning', 'Essentials', 'Dishes and silverware', 'Ethernet connection', 'Hot water', 'Host greets you', 'Garden view', 'Resort access', 'Cooking basics', 'Kitchen', 'BBQ grill', 'Private backyard \\u2013 Fully fenced', 'Long term stays allowed', 'Outdoor dining area', 'Outdoor furniture', 'Wifi', 'Free parking on premises', 'Room-darkening shades', 'Stove', 'Refrigerator', 'Smoke alarm', 'HDTV with Netflix', 'Coffee maker', 'Fire pit', 'Microwave', 'Extra pillows and blankets', 'Bed linens'], 'bathrooms_text': '1.5 baths', 'bedrooms': 1.0, 'latitude': 19.25, 'neighbourhood_cleansed': 'Tlalpan', 'bathrooms': 1.5, 'minimum_nights': 1.0, 'beds': 1.0, 'room_type': 'Entire home/apt'}, page_content='Rustic wooden cabin south of the city in Ajusco located in an ecological reserve neighborhood at the foot of the Xitle volcanoWe have a Roof Garden, barbecue, and a large garden'), Document(id='1447907948774634576', metadata={'price': 1741.0, 'amenities': ['Shampoo', 'Carbon monoxide alarm', 'Conditioner', 'Mini fridge', 'Smoking allowed', 'Hair dryer', 'Shower gel', 'Dishes and silverware', 'Blender', 'Hot water', 'TV', 'Cooking basics', 'Kitchen', 'Toaster', 'BBQ grill', 'Wifi', 'Free parking on premises', 'Stove', 'Smoke alarm', 'Hot water kettle', 'Coffee maker', 'Dining table', 'Coffee', 'Body soap', 'Microwave'], 'bathrooms_text': '1 bath', 'neighbourhood_cleansed': 'Tlalpan', 'beds': 3.0, 'maximum_nights': 365.0, 'bedrooms': 2.0, 'minimum_nights': 2.0, 'review_scores_accuracy': 5.0, 'room_type': 'Entire home/apt', 'has_availability': True, 'longitude': -99.23, 'bathrooms': 1.0, 'latitude': 19.3, 'property_type': 'Entire guest suite'}, page_content='Relax in this rustic cottage with a beautiful garden'), Document(id='1627809771075022607', metadata={'amenities': ['Shampoo', 'Conditioner', 'Hair dryer', 'Freezer', 'Essentials', 'Kitchenette', 'Dishes and silverware', 'Heating', 'Hot water', 'Fire extinguisher', 'Cooking basics', 'Lockbox', 'Air conditioning', 'Clothing storage', 'Outdoor dining area', 'Private entrance', 'Washer', 'Wifi', 'Self check-in', 'Free parking on premises', 'Room-darkening shades', 'Stove', 'Refrigerator', 'Coffee maker', 'Dining table', 'Body soap', 'Bed linens', 'Exterior security cameras on property', 'Free street parking'], 'minimum_nights': 1.0, 'beds': 1.0, 'neighbourhood_cleansed': 'Tlalpan', 'has_availability': True, 'room_type': 'Entire home/apt', 'bathrooms': 1.0, 'price': 1655.0, 'latitude': 19.3, 'maximum_nights': 365.0, 'bathrooms_text': '1 bath', 'bedrooms': 1.0, 'longitude': -99.14, 'review_scores_accuracy': -1.0, 'property_type': 'Entire guest suite'}, page_content='Private residence in the garden of a family home, about a 15-minute walk from Azteca/ Banorte Stadium.'), Document(id='37251212', metadata={'has_availability': True, 'maximum_nights': 1125.0, 'latitude': 19.28, 'minimum_nights': 30.0, 'bathrooms': 2.5, 'longitude': -99.15, 'price': 2375.0, 'room_type': 'Entire home/apt', 'beds': 4.0, 'amenities': ['Essentials', 'Kitchen', 'Hot water', 'Hangers', 'Cooking basics', 'Smoking allowed', 'Dishes and silverware', 'Pets allowed', 'TV', 'Free parking on premises', 'Wifi', 'Dryer', 'Washer'], 'property_type': 'Entire condo', 'review_scores_accuracy': 5.0, 'neighbourhood_cleansed': 'Tlalpan', 'bedrooms': 2.0, 'bathrooms_text': '2.5 baths'}, page_content="Beautiful apartment with ample garden and barbecue. GREAT FOR LONG STAYS!The apartment is located in one of the quietest and safest areas of southern CDMX within a residential complex, very close to important roads such as Tlalpan and Periférico. San Fernando's hospital area is just a few miles away. The subdivision has a corridor, tennis court, basketball court, forest, multi-purpose lounge and park with children's games.")]
"""